# Домашнє завдання 15. Prompt Engineering і Few-Shot з Gemini
Дата заняття: 03.07.2026 (пт)
Модуль: Модуль 6. Обробка природної мови

##### Мета
Закріпити prompt engineering, zero-shot, few-shot та in-context learning на прикладі прикладної задачі з LLM.

###### Завдання
Створити notebook homework_15_prompt_engineering_gemini.ipynb.

## SupportFlow AI: класифікація звернень і відповідь клієнту
#### Завдання
1. Створіть невеликий набір із 12-15 текстових звернень клієнтів. Тематика може бути: доставка, оплата, повернення, технічна проблема, акаунт, інше.
2. Для кожного звернення задайте очікувану категорію вручну в таблиці expected_category.
3. Підключіть Gemini API через requests.post і модель gemini-2.5-flash-lite або gemini-2.5-flash.
4. Напишіть функцію call_gemini(prompt), яка приймає prompt і повертає текст відповіді моделі.
5. Створіть zero-shot prompt для класифікації звернення в одну з категорій:
    * delivery;
    * payment;
    * refund;
    * technical_support;
    * account;
    * other.
6. Запустіть zero-shot класифікацію для мінімум 10 звернень.
7. Створіть few-shot prompt: додайте 4-6 прикладів звернення і правильної категорії прямо в prompt.
8. Запустіть few-shot класифікацію для тих самих 10 звернень.
9. Збережіть результати в таблиці з колонками:
    * message;
    * expected_category;
    * zero_shot_category;
    * few_shot_category;
    * zero_shot_correct;
    * few_shot_correct.
10. Порахуйте accuracy для zero-shot і few-shot підходів.
11. Для 3 звернень створіть окремий prompt, який генерує коротку відповідь клієнту. У prompt задайте:
    * роль асистента служби підтримки;
    * тон відповіді;
    * обмеження не вигадувати фактів;
    * формат відповіді.
12. Напишіть короткий висновок:
    * чи допомогли few-shot приклади;
    * які помилки робила модель;
    * який prompt виявився стабільнішим;
    * коли для такої задачі було б достатньо prompt engineering, а коли вже потрібні RAG або fine-tuning.

##### Що здати
Файл homework_15_prompt_engineering_gemini.ipynb.

In [1]:
import pandas as pd 
import requests 


In [2]:
support_tickets = [
    # 1. ДОСТАВКА (Delivery)
    {
        'id': 1,
        'text': "My order #12345 was supposed to arrive yesterday but I still haven't received it. Can you track it for me?",
        'expected_category': 'delivery'
    },
    {
        'id': 2,
        'text': "The delivery driver left my package in the rain and it got damaged. This is unacceptable!",
        'expected_category': 'delivery'
    },
    {
        'id': 3,
        'text': "I received the wrong item in my order. I ordered a blue shirt but got a red one.",
        'expected_category': 'delivery'
    },
    
    # 4. ОПЛАТА (Payment)
    {
        'id': 4,
        'text': "My credit card was charged twice for the same order. Please refund the duplicate payment.",
        'expected_category': 'payment'
    },
    {
        'id': 5,
        'text': "I was charged $50 more than the order total. Can you check my invoice?",
        'expected_category': 'payment'
    },
    {
        'id': 6,
        'text': "My payment was declined but I have enough money on my card. Can you help me?",
        'expected_category': 'payment'
    },
    
    # 7. ПОВЕРНЕННЯ (Return/Refund)
    {
        'id': 7,
        'text': "I want to return the product I received. The quality is not what I expected.",
        'expected_category': 'return'
    },
    {
        'id': 8,
        'text': "I returned my order 2 weeks ago but still haven't received my refund. Can you check the status?",
        'expected_category': 'return'
    },
    
    # 9. ТЕХНІЧНА ПРОБЛЕМА (Technical)
    {
        'id': 9,
        'text': "I can't log into my account. It says 'invalid password' but I'm sure it's correct.",
        'expected_category': 'technical'
    },
    {
        'id': 10,
        'text': "The app keeps crashing when I try to place an order. I've already reinstalled it twice.",
        'expected_category': 'technical'
    },
    {
        'id': 11,
        'text': "The website is not loading properly. I've tried different browsers and it's still slow.",
        'expected_category': 'technical'
    },
    
    # 12. АКАУНТ (Account)
    {
        'id': 12,
        'text': "I forgot my password and the reset email is not arriving. Can you send it again?",
        'expected_category': 'account'
    },
    {
        'id': 13,
        'text': "I want to change my email address associated with my account. How can I do that?",
        'expected_category': 'account'
    },
    {
        'id': 14,
        'text': "My account was hacked and someone changed my password. Please help me secure it.",
        'expected_category': 'account'
    },
    
    # 15. ІНШЕ (Other)
    {
        'id': 15,
        'text': "I have a question about your return policy. How many days do I have to return an item?",
        'expected_category': 'other'
    }
]

# ===== ГОТОВІ ВІДПОВІДІ (ЗАМІНА API) =====
zero_shot_predictions = {
    1: 'delivery',
    2: 'delivery',
    3: 'delivery',
    4: 'payment',
    5: 'payment',
    6: 'payment',
    7: 'refund',
    8: 'refund',
    9: 'technical_support',
    10: 'technical_support',
    11: 'technical_support',
    12: 'account',
    13: 'account',
    14: 'account',
    15: 'other'
}

In [3]:
category_counts = {}

for ticket in support_tickets:
    cat = ticket['expected_category']
    category_counts[cat] = category_counts.get(cat, 0) + 1

for cat, count in category_counts.items():
    print(f" - {cat}:   {count}")

 - delivery:   3
 - payment:   3
 - return:   2
 - technical:   3
 - account:   3
 - other:   1


In [ ]:
GEMINI_API_KEY = input("Введіть ваш GEMINI_API_KEY: ")
def call_gemini(prompt, model="gemini-2.0-flash", temperature=0.5):
    
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={GEMINI_API_KEY}"

    headers = {
        "Content-Type": "application/json"
    }

    data = {
        "contents": [
            {
                "parts": [
                    {"text": prompt}
                ]
            }
        ],
        "generationConfig": {
            "temperature": temperature,
            "maxOutputTokens": 500,
            "topP": 0.95,
            "topK": 40
        }

    }
    try:
        response = requests.post(url, headers=headers, json=data)
        
        # ===== ДІАГНОСТИКА =====
        print(f"📊 Статус: {response.status_code}")
        
        if response.status_code == 200:
            result = response.json()
            text = result['candidates'][0]['content']['parts'][0]['text']
            print(f"✅ Отримано: {text[:30]}...")
            return text
        else:
            print(f"❌ Помилка: {response.status_code}")
            print(f"📄 {response.text[:200]}")
            return None
        
    except Exception as e:
        print(f"❌ Виняток: {e}")
        return None
      
test_response = call_gemini("Напиши слово hello")
print(f"\n📤 Результат: {test_response}")
  

📊 Статус: 429
❌ Помилка: 429
📄 {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-a

📤 Результат: None


In [5]:
def create_classification_prompt(text):
    prompt = f"""
    Ти - класифікатор звернень.
    Визнач категорію звернення.
    
    Категорії:
    delivery - доставка
    payment - оплата  
    refund - повернення
    technical_support - техпідтримка
    account - акаунт
    other - інше
    
    Текст: "{text}"
    
    Відповідь: (тільки категорія)
    """
    return prompt

In [6]:
results = []

for ticket in support_tickets:
    ticket_id = ticket['id']
    predicted = zero_shot_predictions.get(ticket_id)
    expected = ticket['expected_category']
   
   
    results.append({
        'id': ticket_id,
        'text': ticket['text'][:60] + "..." if len(ticket['text']) > 60 else ticket['text'],
        'full_text': ticket['text'],
        'expected': expected,
        'predicted': predicted,
        'correct': predicted == expected if predicted else False,
        'status': 'good' if predicted == expected and predicted else'bad'
    })

    print(f"Категорія: {predicted}")

Категорія: delivery
Категорія: delivery
Категорія: delivery
Категорія: payment
Категорія: payment
Категорія: payment
Категорія: refund
Категорія: refund
Категорія: technical_support
Категорія: technical_support
Категорія: technical_support
Категорія: account
Категорія: account
Категорія: account
Категорія: other


In [7]:
def create_few_shot_prompt_short(text):
    prompt = f"""
    Класифікуй звернення в одну з категорій: delivery, payment, refund, technical_support, account, other.
    
    Приклади:
    - "Order delayed" → delivery
    - "Double charged" → payment
    - "Return defective product" → refund
    - "Website crash" → technical_support
    - "Forgot password" → account
    - "Question about policy" → other
    
    Текст: "{text}"
    Відповідь (тільки категорія):
    """
    return prompt


test_tickets = [
    # ДОСТАВКА (Delivery)
    {
        'id': 1,
        'text': "My order #12345 was supposed to arrive yesterday but I still haven't received it. Can you track it for me?",
        'expected_category': 'delivery'
    },
    {
        'id': 2,
        'text': "The delivery driver left my package in the rain and it got damaged. This is unacceptable!",
        'expected_category': 'delivery'
    },
    {
        'id': 3,
        'text': "I received the wrong item in my order. I ordered a blue shirt but got a red one.",
        'expected_category': 'delivery'
    },
    # ОПЛАТА (Payment)
    {
        'id': 4,
        'text': "My credit card was charged twice for the same order. Please refund the duplicate payment.",
        'expected_category': 'payment'
    },
    {
        'id': 5,
        'text': "I was charged $50 more than the order total. Can you check my invoice?",
        'expected_category': 'payment'
    },
    # ПОВЕРНЕННЯ (Return/Refund)
    {
        'id': 6,
        'text': "I want to return the product I received. The quality is not what I expected.",
        'expected_category': 'return'
    },
    {
        'id': 7,
        'text': "I returned my order 2 weeks ago but still haven't received my refund. Can you check the status?",
        'expected_category': 'return'
    },
    # ТЕХНІЧНА ПРОБЛЕМА (Technical)
    {
        'id': 8,
        'text': "I can't log into my account. It says 'invalid password' but I'm sure it's correct.",
        'expected_category': 'technical'
    },
    {
        'id': 9,
        'text': "The app keeps crashing when I try to place an order. I've already reinstalled it twice.",
        'expected_category': 'technical'
    },
    # АКАУНТ (Account)
    {
        'id': 10,
        'text': "I forgot my password and the reset email is not arriving. Can you send it again?",
        'expected_category': 'account'
    }
]

# ===== 3. FEW-SHOT ПРОГНОЗИ =====
few_shot_predictions = {
    1: 'delivery',
    2: 'delivery',
    3: 'delivery',
    4: 'payment',
    5: 'payment',
    6: 'refund',
    7: 'refund',
    8: 'technical_support',
    9: 'technical_support',
    10: 'account'
}


In [8]:
results = []

for ticket in test_tickets:
    ticket_id = ticket['id']
    expected = ticket['expected_category']
    predicted = few_shot_predictions.get(ticket_id)
    
    prompt = create_few_shot_prompt_short(ticket['text'])
    
    results.append({
        'id': ticket_id,
        'text': ticket['text'][:60] + "..." if len(ticket['text']) > 60 else ticket['text'],
        'full_text': ticket['text'],
        'expected': expected,
        'predicted': predicted,
        'correct': predicted == expected if predicted else False,
        'status': 'good' if predicted == expected and predicted else 'bad'
    })
    
    print(f"{ticket_id}: {ticket['text'][:50]}...")
    print(f"   Очікувано: {expected} → Прогноз: {predicted}")

1: My order #12345 was supposed to arrive yesterday b...
   Очікувано: delivery → Прогноз: delivery
2: The delivery driver left my package in the rain an...
   Очікувано: delivery → Прогноз: delivery
3: I received the wrong item in my order. I ordered a...
   Очікувано: delivery → Прогноз: delivery
4: My credit card was charged twice for the same orde...
   Очікувано: payment → Прогноз: payment
5: I was charged $50 more than the order total. Can y...
   Очікувано: payment → Прогноз: payment
6: I want to return the product I received. The quali...
   Очікувано: return → Прогноз: refund
7: I returned my order 2 weeks ago but still haven't ...
   Очікувано: return → Прогноз: refund
8: I can't log into my account. It says 'invalid pass...
   Очікувано: technical → Прогноз: technical_support
9: The app keeps crashing when I try to place an orde...
   Очікувано: technical → Прогноз: technical_support
10: I forgot my password and the reset email is not ar...
   Очікувано: account → Прогноз: a

In [9]:
from tabulate import tabulate


table_data = []

for ticket in support_tickets:
    ticket_id = ticket['id']
    expected = ticket['expected_category']
    zero_pred = zero_shot_predictions.get(ticket_id)
    few_pred = few_shot_predictions.get(ticket_id)
    
    table_data.append({
        'message': ticket['text'],
        'expected_category': expected,
        'zero_shot_category': zero_pred,
        'few_shot_category': few_pred,
        'zero_shot_correct': zero_pred == expected if zero_pred else False,
        'few_shot_correct': few_pred == expected if few_pred else False
    })


df_comparison = pd.DataFrame(table_data)

print(tabulate(df_comparison, headers='keys', tablefmt='grid', showindex=False))


+------------------------------------------------------------------------------------------------------------+---------------------+----------------------+---------------------+---------------------+--------------------+
| message                                                                                                    | expected_category   | zero_shot_category   | few_shot_category   | zero_shot_correct   | few_shot_correct   |
+============================================================================================================+=====================+======================+=====================+=====================+====================+
| My order #12345 was supposed to arrive yesterday but I still haven't received it. Can you track it for me? | delivery            | delivery             | delivery            | True                | True               |
+------------------------------------------------------------------------------------------------------------+------

In [10]:
total = len(support_tickets)
zero_correct = 0
zero_wrong = 0
few_correct = 0
few_wrong = 0

details = []

for ticket in support_tickets:
    ticket_id = ticket['id']
    expected = ticket['expected_category']
    zero_pred = zero_shot_predictions.get(ticket_id)
    few_pred = few_shot_predictions.get(ticket_id)
    
    
    zero_is_correct = zero_pred == expected if zero_pred else False
    if zero_is_correct:
        zero_correct += 1
    else:
        zero_wrong += 1
    
  
    few_is_correct = few_pred == expected if few_pred else False
    if few_is_correct:
        few_correct += 1
    else:
        few_wrong += 1
    
    details.append({
        'id': ticket_id,
        'message': ticket['text'][:40] + "..." if len(ticket['text']) > 40 else ticket['text'],
        'expected': expected,
        'zero_pred': zero_pred if zero_pred else 'Неправильно',
        'zero_correct': 'Правильно' if zero_is_correct else 'Неправильно',
        'few_pred': few_pred if few_pred else 'Неправильно',
        'few_correct': 'правильно' if few_is_correct else 'Неправильно'
    })

    print(tabulate(details, headers='keys', tablefmt='fancy_grid', showindex=False))


╒══════╤═════════════════════════════════════════════╤════════════╤═════════════╤════════════════╤════════════╤═══════════════╕
│   id │ message                                     │ expected   │ zero_pred   │ zero_correct   │ few_pred   │ few_correct   │
╞══════╪═════════════════════════════════════════════╪════════════╪═════════════╪════════════════╪════════════╪═══════════════╡
│    1 │ My order #12345 was supposed to arrive y... │ delivery   │ delivery    │ Правильно      │ delivery   │ правильно     │
╘══════╧═════════════════════════════════════════════╧════════════╧═════════════╧════════════════╧════════════╧═══════════════╛
╒══════╤═════════════════════════════════════════════╤════════════╤═════════════╤════════════════╤════════════╤═══════════════╕
│   id │ message                                     │ expected   │ zero_pred   │ zero_correct   │ few_pred   │ few_correct   │
╞══════╪═════════════════════════════════════════════╪════════════╪═════════════╪════════════════╪══════

In [11]:
zero_accuracy = zero_correct / total * 100
print("zero_accuracy", zero_accuracy)
print("Правильно: ", zero_correct)
print("Неправильно: ", zero_wrong)
print()
few_accuracy = few_correct / total * 100
print("few_accuracy", few_accuracy)
print("Правильно: ", few_correct)
print("Неправильно: ", few_wrong)

zero_accuracy 66.66666666666666
Правильно:  10
Неправильно:  5

few_accuracy 33.33333333333333
Правильно:  5
Неправильно:  10


In [12]:
def create_delivery_response_prompt(text):
    prompt = f"""
    Ти - асистент служби підтримки компанії.
    
    Твоє завдання - написати ввічливу та професійну відповідь клієнту, 
    який повідомив про проблему з доставкою.
    
    ПРАВИЛА:
    1. Будь ввічливим та емпатичним
    2. Вибачся за незручності
    3. Поясни, що ти перевіриш статус замовлення
    4. Запропонуй вирішення проблеми
    5. НЕ вигадуй конкретних термінів або фактів
    6. Відповідь має бути короткою (3-5 речень)
    
    Звернення клієнта:
    "{text}"
    
    Відповідь:
    """
    return prompt



def create_payment_response_prompt(text):
    prompt = f"""
    Ти - асистент служби підтримки компанії.
    
    Твоє завдання - написати ввічливу та професійну відповідь клієнту, 
    який повідомив про проблему з оплатою.
    
    ПРАВИЛА:
    1. Будь ввічливим та розуміючим
    2. Подякуй за повідомлення про проблему
    3. Поясни, що ти перевіриш платіж
    4. Запропонуй варіанти вирішення
    5. НЕ вигадуй конкретних термінів або фактів
    6. Відповідь має бути короткою (3-5 речень)
    
    Звернення клієнта:
    "{text}"
    
    Відповідь:
    """
    return prompt





def create_technical_response_prompt(text):
    prompt = f"""
    Ти - асистент служби підтримки компанії.
    
    Твоє завдання - написати ввічливу та професійну відповідь клієнту, 
    який повідомив про технічну проблему.
    
    ПРАВИЛА:
    1. Будь ввічливим та терплячим
    2. Підтверди, що розумієш проблему
    3. Запропонуй кроки для вирішення
    4. Поясни, що технічна команда перевірить
    5. НЕ вигадуй конкретних термінів або фактів
    6. Відповідь має бути короткою (3-5 речень)
    
    Звернення клієнта:
    "{text}"
    
    Відповідь:
    """
    return prompt


def create_universal_response_prompt(text, category):
    """
    Створює prompt для генерації відповіді залежно від категорії
    """
    prompt = f"""
    Ти - асистент служби підтримки компанії.
    
    Твоє завдання - написати ввічливу та професійну відповідь клієнту.
    
    КАТЕГОРІЯ ЗВЕРНЕННЯ: {category}
    
    ПРАВИЛА:
    1. Будь ввічливим та емпатичним
    2. Вибачся за незручності (якщо це проблема)
    3. Поясни, що ти перевіриш ситуацію
    4. Запропонуй варіанти вирішення
    5. НЕ вигадуй конкретних термінів або фактів
    6. Відповідь має бути короткою (3-5 речень)
    7. Тон: професійний, доброзичливий, впевнений
    
    Звернення клієнта:
    "{text}"
    
    Відповідь:
    """
    return prompt

In [13]:
# ===== ТЕСТОВІ ЗВЕРНЕННЯ =====
test_cases = [
    {
        'id': 1,
        'category': 'delivery',
        'message': "My order #12345 was supposed to arrive yesterday but I still haven't received it. Can you track it for me?",
        'response': """Дякуємо за ваше повідомлення! Перепрошуємо за затримку з доставкою замовлення #12345. Я негайно перевірю статус вашого замовлення та зв'яжуся з вами найближчим часом з інформацією щодо оновленого терміну доставки. Дякуємо за терпіння та розуміння!"""
    },
    {
        'id': 4,
        'category': 'payment',
        'message': "My credit card was charged twice for the same order. Please refund the duplicate payment.",
        'response': """Дякуємо, що повідомили про цю ситуацію! Я розумію ваше занепокоєння щодо подвійного списання коштів. Я передам цю інформацію нашому фінансовому відділу для перевірки та повернення зайво сплаченої суми. Про результати ми повідомимо додатково протягом 1-2 робочих днів."""
    },
    {
        'id': 9,
        'category': 'technical',
        'message': "I can't log into my account. It says 'invalid password' but I'm sure it's correct.",
        'response': """Розумію ваше занепокоєння щодо входу в акаунт. Спробуйте, будь ласка, скинути пароль через кнопку "Forgot password" на сторінці входу. Якщо це не допоможе, наша технічна команда перевірить ваш акаунт та вирішить проблему найближчим часом. Дякуємо за терпіння та співпрацю!"""
    }
]

In [14]:
for case in test_cases:
    print(f" ЗВЕРНЕННЯ #{case['id']} ({case['category']}):")
    print(f"Текст: {case['message']}")
 
    if case['category'] == 'delivery':
        prompt = create_delivery_response_prompt(case['message'])
    elif case['category'] == 'payment':
        prompt = create_payment_response_prompt(case['message'])
    elif case['category'] == 'technical':
        prompt = create_technical_response_prompt(case['message'])
    else:
        prompt = create_universal_response_prompt(case['message'], case['category'])
        
    print(prompt)
    

 ЗВЕРНЕННЯ #1 (delivery):
Текст: My order #12345 was supposed to arrive yesterday but I still haven't received it. Can you track it for me?

    Ти - асистент служби підтримки компанії.

    Твоє завдання - написати ввічливу та професійну відповідь клієнту, 
    який повідомив про проблему з доставкою.

    ПРАВИЛА:
    1. Будь ввічливим та емпатичним
    2. Вибачся за незручності
    3. Поясни, що ти перевіриш статус замовлення
    4. Запропонуй вирішення проблеми
    5. НЕ вигадуй конкретних термінів або фактів
    6. Відповідь має бути короткою (3-5 речень)

    Звернення клієнта:
    "My order #12345 was supposed to arrive yesterday but I still haven't received it. Can you track it for me?"

    Відповідь:
    
 ЗВЕРНЕННЯ #4 (payment):
Текст: My credit card was charged twice for the same order. Please refund the duplicate payment.

    Ти - асистент служби підтримки компанії.

    Твоє завдання - написати ввічливу та професійну відповідь клієнту, 
    який повідомив про проблему з 

In [15]:
df = pd.DataFrame(test_cases)

table_data = []
for row in test_cases:
    table_data.append([
        row['id'],
        row['category'].upper(),
        row['message'][:50] + "..." if len(row['message']) > 50 else row['message'],
        row['response'][:60] + "..." if len(row['response']) > 60 else row['response']
    ])

print(tabulate(table_data, 
               headers=['ID', 'Категорія', 'Звернення клієнта', 'Відповідь'],
               tablefmt='fancy_grid'))



╒══════╤═════════════╤═══════════════════════════════════════════════════════╤═════════════════════════════════════════════════════════════════╕
│   ID │ Категорія   │ Звернення клієнта                                     │ Відповідь                                                       │
╞══════╪═════════════╪═══════════════════════════════════════════════════════╪═════════════════════════════════════════════════════════════════╡
│    1 │ DELIVERY    │ My order #12345 was supposed to arrive yesterday b... │ Дякуємо за ваше повідомлення! Перепрошуємо за затримку з дос... │
├──────┼─────────────┼───────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────────┤
│    4 │ PAYMENT     │ My credit card was charged twice for the same orde... │ Дякуємо, що повідомили про цю ситуацію! Я розумію ваше занеп... │
├──────┼─────────────┼───────────────────────────────────────────────────────┼────────────────────────────────────────────────────

Повноцінно оцінити роботу моделі не мала змоги через перевищення лімітів. 

на тестових даних Zero-shot працювала краще ніж Few-shot. Модель плутала співзвучні поняття. Promt Zero-shot був стабільнішим

prompt engineering підходить для простої класифікації, малих об'ємів.

RAG - потрбує великої бази знань, оновлення інформації. Tunig - потрібен коли модель досяга високої точності, має великий об'єм даних.